In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# ============================================================
# CONFIGURACIÓN
# ============================================================
# Configuración de rutas (asegúrate de que ROOT suba a la raíz)

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from config.paths import RAW_DIR, PROCESSED_DIR

FACTOR_IVA = 0.16 / 1.16   # 0.137931...

# ------------------------------------------------------------
# FUNCIONES DE CLASIFICACIÓN DE CLAVES GRAVADAS
# ------------------------------------------------------------
def es_gravado_2018(clave: str) -> bool:
    """
    True si la clave de gasto ENIGH 2018 está gravada para IVA al 16%.
    """
    # Alimentos preparados fuera del hogar
    if clave in ('A198','A199','A200','A201','A202','A243','A244','A245','A246','A247'):
        return True
    # Bebidas con azúcar, concentrados, polvos
    if clave in ('A218','A219','A220','A221','A222'):
        return True
    # Bebidas alcohólicas y tabaco
    if 'A223' <= clave <= 'A241':
        return True

    # Transporte público terrestre: tasa 0 %
    if clave.startswith('B'):
        return False

    # Limpieza y menaje (C) – gravado
    if clave.startswith('C'):
        return True

    # Cuidado personal y cosméticos (D) – gravado
    if clave.startswith('D'):
        return True

    # Educación
    if clave.startswith('E'):
        # Servicios educativos: exentos
        if clave in ('E001','E002','E003','E004','E005','E006','E007','E008','E009','E010',
                     'E011','E012','E013','E018','E020'):
            return False
        # Libros, periódicos, revistas: tasa 0 %
        if clave in ('E022','E023','E024'):
            return False
        # Espectáculos, cines, teatros: gravado
        if 'E027' <= clave <= 'E034':
            return True
        # Resto de material educativo (papelería, equipo, imprevistos): gravado
        return True

    # Comunicaciones y combustibles (F) – gravado
    if clave.startswith('F'):
        return True

    # Vivienda (G)
    if clave.startswith('G'):
        # Alquiler y cuotas: exentos
        if clave.startswith('G0') or clave.startswith('G1'):  # G001-G106
            return False
        # Combustibles para cocinar/calentar (gas, petróleo, diesel)
        if clave in ('G009','G010','G011','G014'):
            return True
        # Leña, carbón, velas: tasa 0 %
        return False

    # Vestido y calzado (H) – gravado
    if clave.startswith('H'):
        return True

    # Muebles, blancos, utensilios, electrodomésticos (I, K) – gravado
    if clave.startswith('I') or clave.startswith('K'):
        return True

    # Salud (J)
    if clave.startswith('J'):
        # Aparatos ortopédicos, lentes, etc. (gravados)
        if clave in ('J065','J066','J067'):
            return True
        # Medicamentos y servicios: exentos o tasa 0 %
        return False

    # Esparcimiento y electrónica (L) – gravado
    if clave.startswith('L'):
        return True

    # Transporte (M)
    if clave.startswith('M'):
        # Transporte terrestre foráneo: tasa 0 %
        if clave in ('M001','M002'):
            return False
        # Vehículos, refacciones, mantenimiento, aéreo: gravado
        return True

    # Otros servicios (N) – gravado
    if clave.startswith('N'):
        return True

    # Financiero (Q) – exento
    if clave.startswith('Q'):
        return False

    # Servicios del hogar (R)
    if clave.startswith('R'):
        # Agua y predial: exentos
        if clave in ('R002','R004'):
            return False
        # Electricidad, gas, telefonía, TV, internet, tenencia: gravado
        return True

    # Por defecto, para cualquier clave nueva: asumimos gravado
    return True


def es_gravado_2024(clave: str) -> bool:
    """
    True si la clave de gasto ENIGH 2024 (6 dígitos) está gravada para IVA al 16%.
    """
    # Alimentos preparados fuera del hogar (1111xx)
    if clave.startswith('1111'):
        return True

    # Resto de alimentos y bebidas no alcohólicas (01xxxx)
    if clave.startswith('01'):
        # Bebidas gravadas (refrescos, jugos azucarados, energéticas, concentrados)
        if clave in ('012601','012602','012604','012903','012904','012905','012100'):
            return True
        # Alimentos básicos y agua natural (012501): tasa 0%
        return False

    # Bebidas alcohólicas y tabaco (02xxxx)
    if clave.startswith('02'):
        return True

    # Vestido y calzado (03xxxx)
    if clave.startswith('03'):
        return True

    # Vivienda (04xxxx)
    if clave.startswith('04'):
        # Alquiler: exento
        if clave in ('041104','041211','041221'):
            return False
        # Electricidad, gas LP, gas natural: gravados
        if clave.startswith('0451') or clave.startswith('045221') or clave.startswith('045222'):
            return True
        # Agua (044110): exenta
        # Otros (recolección basura, vigilancia, etc.): según caso, en general gravados si son servicios privados
        # Para ser conservadores, los consideramos gravados excepto los marcados explícitamente exentos.
        return True

    # Muebles, electrodomésticos, menaje (05xxxx) – gravado
    if clave.startswith('05'):
        return True

    # Salud (06xxxx)
    if clave.startswith('06'):
        # Medicamentos y productos farmacéuticos (0611xx): tasa 0 %
        if clave.startswith('0611'):
            return False
        # Servicios médicos y hospitalarios (062xxx, 063xxx, 064xxx): exentos
        if clave.startswith('062') or clave.startswith('063') or clave.startswith('064'):
            return False
        # Aparatos y dispositivos médicos (0613xx): normalmente gravados
        return True

    # Transporte (07xxxx)
    if clave.startswith('07'):
        # Transporte público terrestre (073xxx): tasa 0 %
        if clave.startswith('073'):
            return False
        # Vehículos, combustibles, mantenimiento: gravados
        return True

    # Comunicaciones (08xxxx) – gravado
    if clave.startswith('08'):
        return True

    # Recreación y cultura (09xxxx)
    if clave.startswith('09'):
        # Libros, periódicos, revistas (tasa 0 %)
        if clave in ('097111','097210','097220'):
            return False
        # El resto: cines, conciertos, museos, equipo de sonido, juguetes: gravado
        return True

    # Educación (10xxxx): servicios exentos
    if clave.startswith('10'):
        return False

    # Restaurantes y alojamiento (11xxxx) ya cubiertos; alojamiento (112xxx) puede estar gravado
    if clave.startswith('112'):
        return True

    # Seguros (12xxxx): gravado
    if clave.startswith('12'):
        return True

    # Cuidado personal y otros servicios (13xxxx)
    if clave.startswith('13'):
        return True

    # Otras erogaciones (17xxxx, 18xxxx)
    if clave.startswith('17') or clave.startswith('18'):
        # Educación inicial (174xxx): exenta, pero por simplicidad asumimos el resto gravado
        return True

    # Por defecto: gravado
    return True


# ------------------------------------------------------------
# PROCESAMIENTO PRINCIPAL
# ------------------------------------------------------------
frames_salida = []

for year in [2018, 2024]:
    print(f"Procesando ENIGH {year} – gastospersona...")

    # ---- Carga de datos con tipos explícitos ----
    archivo = RAW_DIR / f"gastospersona_{year}.csv"
    df = pd.read_csv(
        archivo,
        dtype={
            "folioviv": "string",
            "foliohog": "string",
            "numren": "string",      # tratamos como string para el merge
            "clave": "string",
            "tipo_gasto": "string"
        },
        low_memory=False
    )

    # ---- Filtrar solo gastos monetarios ----
    df = df[df["tipo_gasto"] == "G1"].copy()

    # ---- Asegurar que las columnas numéricas se lean correctamente ----
    # 'gasto_tri' puede venir como cadena o numérico, lo forzamos a float
    df["gasto_tri"] = pd.to_numeric(df["gasto_tri"], errors="coerce").fillna(0.0)

    # ---- Clasificar si el gasto lleva IVA (gravado) ----
    if year == 2018:
        df["es_gravado"] = df["clave"].apply(es_gravado_2018)
    else:
        df["es_gravado"] = df["clave"].apply(es_gravado_2024)

    # ---- Filtrar informalidad (solo si existe la columna 'lugar_comp') ----
    if "lugar_comp" in df.columns:
        # Convertir a numérico por si acaso
        df["lugar_comp"] = pd.to_numeric(df["lugar_comp"], errors="coerce").fillna(0).astype(int)
        informal_locs = [1, 2, 3, 17]   # códigos de informalidad
        df["es_formal"] = ~df["lugar_comp"].isin(informal_locs)
        print(f"  -> Filtro de lugar de compra aplicado. Códigos informales: {informal_locs}")
    else:
        # Si no existe la variable, asumimos que todas las compras son formales
        df["es_formal"] = True
        print("  -> Variable 'lugar_comp' no encontrada. Se asume que todos los gastos son formales.")

    # ---- Cálculo del IVA incluido en el precio ----
    df["iva"] = np.where(
        df["es_gravado"] & df["es_formal"],
        df["gasto_tri"] * FACTOR_IVA,
        0.0
    )

    # ---- Agrupar por persona (folioviv, foliohog, numren) ----
    persona = df.groupby(["folioviv", "foliohog", "numren"], as_index=False).agg(
        gasto_total = ("gasto_tri", "sum"),
        gasto_iva   = ("iva", "sum")
    )
    persona["year"] = year
    frames_salida.append(persona)

# ------------------------------------------------------------
# UNIR AÑOS Y EXPORTAR
# ------------------------------------------------------------
final = pd.concat(frames_salida, ignore_index=True)
final = final[["folioviv", "foliohog", "numren", "year", "gasto_total", "gasto_iva"]]


  # Guardar temporal
out_file = PROCESSED_DIR / 'iva_persona_2018_2024.csv'
final.to_csv(out_file, index=False)
print(f"Archivo guardado en: {out_file}")

Procesando ENIGH 2018 – gastospersona...
  -> Variable 'lugar_comp' no encontrada. Se asume que todos los gastos son formales.
Procesando ENIGH 2024 – gastospersona...
  -> Variable 'lugar_comp' no encontrada. Se asume que todos los gastos son formales.
Archivo guardado en: /Users/jesussanchez/Documents/Pre-trabajo/3.-Mapping-the-Fiscal-Burden/fiscal_incidence_project/data/processed/iva_persona_2018_2024.csv
